In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/home/m.gromadzki/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "/storage/models/Qwen__Qwen3-14B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 443/443 [00:06<00:00, 65.02it/s, Materializing param=model.norm.weight]                              


In [3]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=8192 * 4,
        do_sample=False
    )
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [12]:
import json
from tqdm import tqdm

# Load saved tasks
with open("tasks5.json", "r") as f:
    tasks = json.load(f)

In [13]:
results = []
total = 0
correct = 0

def build_few_shot_prompt(train_examples, test_sentence):
    """
    Combine all train examples into a single message, then append test sentence.
    """
    # Format train examples as lines
    train_text = "\n".join([f"{en} -> {isl}" for en, isl in train_examples])

    messages = [
        {"role": "system", "content": "You are a helpful assistant that translates English sentences into ISL (invented sign language)."},
        {"role": "user", "content": f"Here are some example translations:\n{train_text}\n\nTranslate this sentence:"},
        {"role": "user", "content": test_sentence}
    ]
    return messages

# Loop over all tasks
for task_idx, task in enumerate(tqdm(tasks[:30], desc="Tasks")):
    train_examples = task["train"]
    test_examples = task["test"]

    for test_en, test_isl in test_examples:
        messages = build_few_shot_prompt(train_examples, test_en)

        # Tokenize
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        # Generate
        with torch.inference_mode():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=8192 * 4,
                do_sample=False
            )

        # Remove input tokens
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        # Decode
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        generated_text = response.split("</think>")[-1].strip()
        # Exact match
        is_correct = generated_text == test_isl
        if is_correct:
            correct += 1
        total += 1

        results.append({
            "task_idx": task_idx,
            "input": test_en,
            "expected": test_isl,
            "predicted": generated_text,
            "correct": is_correct,
            "response": response
        })

Tasks: 100%|██████████| 30/30 [19:18<00:00, 38.60s/it]


In [14]:
accuracy = correct / total
print(f"Overall test set exact match accuracy: {accuracy:.2%}")

Overall test set exact match accuracy: 13.33%


In [7]:
results

[{'task_idx': 0,
  'input': 'low son not left same truths',
  'expected': 'iqo xapifag efarhj bel devit iev',
  'predicted': 'iqo xapifag efarhj bel devit seosar',
  'correct': False,
  'response': '<think>\nOkay, let\'s tackle this translation. The user wants to translate "low son not left same truths" into ISL. First, I need to break down the sentence into its components.\n\n"Low son" – "low" here might be an adjective describing "son." In ISL, adjectives often come before the noun. So "low son" could be "iqo xapifag" since "iqo" is low and "xapifag" is son. \n\n"Not left" – The verb "left" in this context might be "left behind" or "not remaining." In ISL, "not left" could be "efarhj bel" where "efarhj" is "not" and "bel" is "left." \n\n"Same truths" – "Same" is "devit" and "truths" is "seosar." So "same truths" would be "devit seosar."\n\nPutting it all together: "iqo xapifag efarhj bel devit seosar." Let me check if the word order makes sense. In ISL, the structure is usually subje

In [ ]:
# Qwen__Qwen3-14B
# lvl1 - 97%
# lvl2 - 60%
# lvl3 - 47%
# lvl4 - 33%
# lvl5 - 13%